# R3-CAP · Capstone — Profundización (opcional) 🔬

**Formación Pública — Rama R3 · IA Aplicada · Notebook de profundización**

Opcional. Tres mejoras que separan una demo de un servicio en producción:

1. **Calibrar el umbral** de abstención con evidencia (preguntas dentro y fuera del corpus).
2. **Filtrar por metadatos**: si sabes el tema de la consulta, busca solo ahí.
3. **Bitácora de auditoría** anonimizada: registrar cada consulta sin guardar datos personales.

Todo corre sin red ni LLM. Usa el corpus de fichas; cámbialo por el tuyo igual que en la lección.

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import os, re, json, urllib.request
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

CORPUS_CSV = "fichas.csv"
if not os.path.exists(CORPUS_CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{CORPUS_CSV}", CORPUS_CSV)
corpus = pd.read_csv(CORPUS_CSV)[["id", "titulo", "tema", "url", "texto"]]

def trocear(texto, max_palabras=120):
    """Corta en cada subtítulo '## ' o al pasar de max_palabras (la función de R3-03)."""
    trozos, actual, n = [], [], 0
    for linea in str(texto).split("\n"):
        if not linea.strip():                    # las líneas en blanco no aportan
            continue
        palabras = len(linea.split())
        if actual and (linea.startswith("## ") or n + palabras > max_palabras):
            trozos.append("\n".join(actual))
            actual, n = [], 0
        actual.append(linea)
        n += palabras
    if actual:
        trozos.append("\n".join(actual))
    return trozos

trozos = pd.DataFrame([{"id": d.id, "titulo": d.titulo, "tema": d.tema, "url": d.url, "texto": t}
                       for d in corpus.itertuples() for t in trocear(d.texto)])
vect = TfidfVectorizer(strip_accents="unicode", analyzer="char_wb", ngram_range=(3, 5))
X = vect.fit_transform((trozos["titulo"] + "\n") * 3 + trozos["texto"])

def puntajes(pregunta):
    return (X @ vect.transform([pregunta]).T).toarray().ravel()

def recuperar(pregunta, k=3):
    p = puntajes(pregunta)
    mejores = np.argsort(-p, kind="stable")[:k]
    res = trozos.iloc[mejores].copy()
    res["puntaje"] = p[mejores]
    return res

# Tus preguntas de evaluación: (pregunta, id del documento que DEBERÍA aparecer).
# Escríbelas ANTES de mirar resultados. Estas son de ejemplo para el corpus de fichas:
# reemplázalas por preguntas reales de las personas que atiende tu servicio.
MIS_PREGUNTAS = [
    ("¿Cuándo se paga el aguinaldo de Fiestas Patrias a los pensionados?", 26553),
    ("¿Qué es la Pensión Garantizada Universal y cuánto aumentó?", 130457),
    ("¿Cómo cambio mi domicilio si soy pensionado del IPS y vivo afuera?", 6347),
    ("¿Cómo pido asesoría previsional gratuita?", 9648),
    ("¿Cómo obtengo el certificado de alumno prioritario?", 20613),
    ("¿Qué es el Bolsillo Familiar Electrónico?", 115303),
    ("¿Cómo postulo al Capital Semilla Emprende?", 20608),
    ("¿Cómo pido la Beca Bicentenario?", 2227),
    ("¿Quién recibe el Aporte Familiar Permanente?", 38913),
    ("¿Cómo me inscribo en la Bolsa Nacional de Empleo?", 5072),
    ("Me rechazaron la licencia porque dicen que no hice reposo, ¿qué hago?", 3976),
    ("¿Hay ayuda para pagar la casa de reposo de mi mamá?", 5039),
]

# Preguntas que el asistente NO debería responder (fuera del corpus)
FUERA_DEL_CORPUS = [
    "¿Cómo se prepara una empanada de pino?", "¿Quién ganó la Copa América 2015?",
    "¿Cuál es la capital de Francia?", "¿Cuánto cuesta un kilo de pan?",
    "¿Qué tiempo hará mañana?", "¿Cuál es el horario del metro?",
    "¿Cómo renuevo el pasaporte?", "¿Quién es el presidente de Argentina?",
]
print("Trozos:", len(trozos), "| preguntas dentro:", len(MIS_PREGUNTAS), "| fuera:", len(FUERA_DEL_CORPUS))

## 1. Calibrar el umbral con evidencia

El umbral decide cuándo abstenerse. Muy bajo: responde cosas que no sabe. Muy alto: se abstiene ante preguntas legítimas. Para cada umbral candidato, cuenta las **decisiones correctas**: responder (mejor puntaje ≥ umbral) las de `MIS_PREGUNTAS` y abstenerse (< umbral) en las de `FUERA_DEL_CORPUS`. Quédate con el que más acierta.

### ✍️ Ejercicio 1 — Construye la tabla de calibración

In [ ]:
mejor_dentro = [puntajes(p).max() for p, _ in MIS_PREGUNTAS]
mejor_fuera = [puntajes(p).max() for p in FUERA_DEL_CORPUS]
filas = []
for u in [0.10, 0.20, 0.30, 0.40]:
    responde_bien = sum(s >= u for s in mejor_dentro)
    se_abstiene_bien = sum(s < u for s in mejor_fuera)
    filas.append({"umbral": u, "responde_bien": responde_bien, "se_abstiene_bien": se_abstiene_bien,
                  "aciertos": responde_bien + se_abstiene_bien})
calibracion = pd.DataFrame(filas)
umbral_elegido = calibracion.loc[calibracion["aciertos"].idxmax(), "umbral"]
print(calibracion); print("umbral elegido:", umbral_elegido)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert list(calibracion["umbral"]) == [0.10, 0.20, 0.30, 0.40]
    for f in calibracion.itertuples():
        assert f.responde_bien == sum(s >= f.umbral for s in mejor_dentro)
        assert f.se_abstiene_bien == sum(s < f.umbral for s in mejor_fuera)
    assert umbral_elegido == calibracion.loc[calibracion["aciertos"].idxmax(), "umbral"]
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'responde_bien = sum(s >= u for s in mejor_dentro); se_abstiene_bien = sum(s < u for s in mejor_fuera); umbral_elegido = calibracion.loc[calibracion["aciertos"].idxmax(), "umbral"].')
    print("   Detalle:", e)

## 2. Filtrar por metadatos (tema)

Si tu asistente vive en la página de **un** tema (por ejemplo, «Chilenos en el exterior»), no tiene sentido buscar en todo el corpus: un filtro por metadatos evita que gane un trozo parecido pero de otro tema. Escribe `recuperar_en_tema(pregunta, temas, k)` que solo considere trozos cuyo `tema` esté en `temas`.

### ✍️ Ejercicio 2 — Escribe `recuperar_en_tema`

In [ ]:
def recuperar_en_tema(pregunta, temas, k=3):
    p = puntajes(pregunta)
    p = np.where(trozos["tema"].isin(temas), p, -1.0)       # fuera del tema: descartado
    mejores = np.argsort(-p, kind="stable")[:k]
    res = trozos.iloc[mejores].copy()
    res["puntaje"] = p[mejores]
    return res

pregunta = "¿Cómo saco el certificado de antecedentes?"
print(recuperar(pregunta, 1)[["id", "titulo", "tema"]])                                  # todo el corpus
print(recuperar_en_tema(pregunta, ["chilenos-en-el-exterior"], 1)[["id", "titulo", "tema"]])  # solo ese tema

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    r = recuperar_en_tema("¿Cómo saco el certificado de antecedentes?", ["chilenos-en-el-exterior"])
    assert len(r) == 3 and set(r["tema"]) == {"chilenos-en-el-exterior"}, "solo deben volver trozos del tema pedido"
    assert r["id"].iloc[0] == 49064, "en ese tema, la ficha correcta es la del certificado para chilenos en el extranjero"
    r2 = recuperar_en_tema("¿Cómo me afilio a Fonasa?", ["salud", "certificados"], k=5)
    assert set(r2["tema"]) <= {"salud", "certificados"} and (r2["puntaje"] >= 0).all()
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'p = np.where(trozos["tema"].isin(temas), p, -1.0).')
    print("   Detalle:", e)

## 3. Bitácora de auditoría, sin datos personales

Un servicio en producción **registra** cada consulta: para auditar, medir y mejorar. Pero la bitácora no puede convertirse en una base de datos personales. Escribe `anonimizar(texto)` (reemplaza RUT por `[RUT]` y correos por `[CORREO]`) y `registrar(ruta, pregunta, resultado)`, que agrega **una línea JSON** con fecha, pregunta anonimizada, si se abstuvo y las fuentes.

### ✍️ Ejercicio 3 — Escribe `anonimizar` y `registrar`

In [ ]:
def anonimizar(texto):
    texto = re.sub(r"\b\d{1,2}\.?\d{3}\.?\d{3}-[\dkK]\b", "[RUT]", texto)
    return re.sub(r"\b[\w.+-]+@[\w-]+\.[\w.]+\b", "[CORREO]", texto)

def registrar(ruta, pregunta, resultado):
    linea = {"fecha": datetime.now().isoformat(timespec="seconds"),
             "pregunta": anonimizar(pregunta),
             "abstuvo": not resultado["fuentes"],
             "fuentes": resultado["fuentes"]}
    with open(ruta, "a", encoding="utf-8") as f:
        f.write(json.dumps(linea, ensure_ascii=False) + "\n")

BITACORA = "bitacora_demo.jsonl"
if os.path.exists(BITACORA):
    os.remove(BITACORA)
registrar(BITACORA, "Soy 12.345.678-5, escríbanme a ana@correo.cl: ¿cómo pido la PGU?",
          {"respuesta": "...", "fuentes": ["https://www.chileatiende.gob.cl/fichas/130457"]})
registrar(BITACORA, "¿Quién ganó la Copa América 2015?", {"respuesta": "No encontré…", "fuentes": []})
print(open(BITACORA, encoding="utf-8").read())

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert anonimizar("Soy 12.345.678-5 y mi correo es ana@correo.cl") == "Soy [RUT] y mi correo es [CORREO]"
    if os.path.exists(BITACORA):
        os.remove(BITACORA)
    registrar(BITACORA, "Mi RUT es 9.876.543-K, ¿cómo pido la PGU?", {"respuesta": "x", "fuentes": ["u1"]})
    registrar(BITACORA, "¿Quién ganó la Copa América 2015?", {"respuesta": "y", "fuentes": []})
    lineas = [json.loads(l) for l in open(BITACORA, encoding="utf-8")]
    assert len(lineas) == 2, "debe haber una línea por consulta"
    assert "9.876.543-K" not in open(BITACORA, encoding="utf-8").read(), "¡la bitácora guardó un RUT!"
    assert lineas[0]["abstuvo"] is False and lineas[1]["abstuvo"] is True
    assert {"fecha", "pregunta", "abstuvo", "fuentes"} <= set(lineas[0])
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 're.sub(PATRON_RUT, "[RUT]", texto) y luego el correo; en registrar: f.write(json.dumps(linea, ensure_ascii=False) + "\\n") con open(ruta, "a").')
    print("   Detalle:", e)

## Para llevar

- El **umbral** no se adivina: se calibra con preguntas dentro y fuera del corpus, y se
  recalibra cuando cambia el corpus o el índice.
- Los **metadatos** (tema, institución, fecha de vigencia) son filtros baratos y poderosos.
- Una **bitácora** permite auditar y mejorar el asistente; anonimizar **antes** de guardar es
  parte del diseño, no un detalle. Las expresiones regulares son una primera barrera: revisa
  qué otros datos personales podrían llegar (direcciones, diagnósticos, nombres).

---

*Fin de la profundización del capstone R3.*